# ПЗ 4.1.1. Диагностика узких мест при обработке больших наборов данных на одном компьютере

**Дисциплина:** Системы обработки больших данных  
**Тема:** Диагностика узких мест при обработке больших наборов данных на одном компьютере  
**Цель работы:** сравнить полную загрузку файла в память, пофайловую обработку и блочную обработку, а затем сделать вывод о том, где именно возникают узкие места: память, время, формат файла, число признаков и количество проходов по данным.

**Выполнил:** Дмитрий Трифонов, 12-25РПм, 1 курс, магистр, Программная инженерия

В этой работе используются учебные синтетические датасеты. Они достаточно велики, чтобы показать различия между подходами, но при этом безопасны для запуска на обычном учебном компьютере.

## 1. Что должен показать результат практической работы

После выполнения работы студент должен уметь:

1. различать полную загрузку, пофайловую и блочную обработку;
2. измерять время выполнения и приблизительное потребление памяти;
3. объяснять, почему один и тот же набор данных ведёт себя по-разному в зависимости от способа обработки;
4. делать инженерный вывод, когда допустима полная загрузка, а когда нужно переходить к потоковой или блочной схеме.

In [17]:
from pathlib import Path
import json
import gc
import time
import os
import pandas as pd
import numpy as np

try:
    import psutil
except ImportError:
    psutil = None

BASE_DIR = Path('.')
with open(BASE_DIR / '12-1_variant_specs.json', 'r', encoding='utf-8') as f:
    VARIANTS = {item['variant']: item for item in json.load(f)}

print(f'Рабочий каталог: {BASE_DIR.resolve()}')
print(f'Доступно вариантов: {len(VARIANTS)}')

Рабочий каталог: C:\Users\admin\Desktop\test
Доступно вариантов: 1


## 2. Выбор варианта

Установите номер варианта.  
Если преподаватель не назначил вариант отдельно, можно начать с **варианта По списку в журнале**.

In [18]:
VARIANT = 12
cfg = VARIANTS[VARIANT]
cfg

{'variant': 12,
 'full_file': '12-2_sales_small.csv',
 'chunk_file': '12-3_events_text.csv',
 'chunksize': 8000,
 'group_col': 'region',
 'metric': 'revenue',
 'folder': './',
 'question': 'Сравнить небольшой числовой файл и текстовый файл среднего размера; объяснить, почему размер на диске и память в процессе — не одно и то же.'}

## 3. Вспомогательные функции

В работе будем фиксировать:
- время выполнения;
- изменение занимаемой памяти процессом;
- размер файла на диске;
- простые статистики по набору данных.

> Замечание: измерение памяти в учебной работе является ориентировочным. Нам важна сравнительная картина, а не абсолютная точность до байта.

In [19]:
def rss_mb():
    if psutil is None:
        return np.nan
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)

def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 ** 2)

def summarize_frame(df, group_col=None, metric=None):
    result = {
        'rows': len(df),
        'cols': len(df.columns),
        'missing_total': int(df.isna().sum().sum())
    }
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    result['numeric_cols'] = len(numeric_cols)
    if group_col is not None and metric is not None and group_col in df.columns and metric in df.columns:
        grouped = df.groupby(group_col)[metric].agg(['count', 'mean', 'sum']).reset_index()
        result['grouped'] = grouped
    return result

def show_grouped(grouped, max_rows=10):
    if grouped is None:
        print('Групповая статистика не вычислялась.')
    else:
        display(grouped.head(max_rows))

def timed_full_read(path, group_col=None, metric=None):
    mem_before = rss_mb()
    t0 = time.perf_counter()
    df = pd.read_csv(path)
    summary = summarize_frame(df, group_col, metric)
    elapsed = time.perf_counter() - t0
    mem_after = rss_mb()
    result = {
        'method': 'full_read',
        'path': str(path),
        'file_size_mb': round(file_size_mb(path), 2),
        'elapsed_sec': round(elapsed, 4),
        'rss_before_mb': round(mem_before, 2) if not np.isnan(mem_before) else np.nan,
        'rss_after_mb': round(mem_after, 2) if not np.isnan(mem_after) else np.nan,
        'rss_delta_mb': round(mem_after - mem_before, 2) if not np.isnan(mem_before) and not np.isnan(mem_after) else np.nan,
        'rows': summary['rows'],
        'cols': summary['cols'],
        'missing_total': summary['missing_total'],
        'numeric_cols': summary['numeric_cols']
    }
    grouped = summary.get('grouped')
    return result, grouped, df

def timed_chunk_read(path, chunksize, group_col=None, metric=None):
    mem_before = rss_mb()
    t0 = time.perf_counter()
    total_rows = 0
    total_missing = 0
    cols = None
    numeric_cols = None
    aggregated = []
    for chunk in pd.read_csv(path, chunksize=chunksize):
        total_rows += len(chunk)
        total_missing += int(chunk.isna().sum().sum())
        cols = len(chunk.columns)
        numeric_cols = len(chunk.select_dtypes(include='number').columns)
        if group_col is not None and metric is not None and group_col in chunk.columns and metric in chunk.columns:
            part = chunk.groupby(group_col)[metric].agg(['count','sum']).reset_index()
            aggregated.append(part)
    elapsed = time.perf_counter() - t0
    mem_after = rss_mb()
    grouped = None
    if aggregated:
        grouped = pd.concat(aggregated, ignore_index=True).groupby(group_col)[['count','sum']].sum().reset_index()
        grouped['mean'] = grouped['sum'] / grouped['count']
        grouped = grouped[[group_col, 'count', 'mean', 'sum']]
    result = {
        'method': 'chunk_read',
        'path': str(path),
        'file_size_mb': round(file_size_mb(path), 2),
        'chunksize': chunksize,
        'elapsed_sec': round(elapsed, 4),
        'rss_before_mb': round(mem_before, 2) if not np.isnan(mem_before) else np.nan,
        'rss_after_mb': round(mem_after, 2) if not np.isnan(mem_after) else np.nan,
        'rss_delta_mb': round(mem_after - mem_before, 2) if not np.isnan(mem_before) and not np.isnan(mem_after) else np.nan,
        'rows': total_rows,
        'cols': cols,
        'missing_total': total_missing,
        'numeric_cols': numeric_cols
    }
    return result, grouped

def timed_folder_read(folder_path, group_col=None, metric=None):
    mem_before = rss_mb()
    t0 = time.perf_counter()
    total_rows = 0
    total_missing = 0
    cols = None
    numeric_cols = None
    aggregated = []
    folder = Path(folder_path)
    files = sorted(folder.glob('*.csv'))
    file_table = []
    for path in files:
        df = pd.read_csv(path)
        total_rows += len(df)
        total_missing += int(df.isna().sum().sum())
        cols = len(df.columns)
        numeric_cols = len(df.select_dtypes(include='number').columns)
        file_table.append({'file': path.name, 'rows': len(df), 'size_mb': round(file_size_mb(path), 2)})
        if group_col is not None and metric is not None and group_col in df.columns and metric in df.columns:
            part = df.groupby(group_col)[metric].agg(['count','sum']).reset_index()
            aggregated.append(part)
        del df
        gc.collect()
    elapsed = time.perf_counter() - t0
    mem_after = rss_mb()
    grouped = None
    if aggregated:
        grouped = pd.concat(aggregated, ignore_index=True).groupby(group_col)[['count','sum']].sum().reset_index()
        grouped['mean'] = grouped['sum'] / grouped['count']
        grouped = grouped[[group_col, 'count', 'mean', 'sum']]
    result = {
        'method': 'file_by_file',
        'path': str(folder_path),
        'files_count': len(files),
        'elapsed_sec': round(elapsed, 4),
        'rss_before_mb': round(mem_before, 2) if not np.isnan(mem_before) else np.nan,
        'rss_after_mb': round(mem_after, 2) if not np.isnan(mem_after) else np.nan,
        'rss_delta_mb': round(mem_after - mem_before, 2) if not np.isnan(mem_before) and not np.isnan(mem_after) else np.nan,
        'rows': total_rows,
        'cols': cols,
        'missing_total': total_missing,
        'numeric_cols': numeric_cols
    }
    return result, grouped, pd.DataFrame(file_table)

def cleanup(*objects):
    for obj in objects:
        try:
            del obj
        except:
            pass
    gc.collect()
    return 'Промежуточные объекты удалены, сборщик мусора вызван.'

## 4. Краткое описание варианта и ожидаемых файлов

In [20]:
cfg

{'variant': 12,
 'full_file': '12-2_sales_small.csv',
 'chunk_file': '12-3_events_text.csv',
 'chunksize': 8000,
 'group_col': 'region',
 'metric': 'revenue',
 'folder': './',
 'question': 'Сравнить небольшой числовой файл и текстовый файл среднего размера; объяснить, почему размер на диске и память в процессе — не одно и то же.'}

In [21]:
full_path = BASE_DIR / cfg['full_file']
chunk_path = BASE_DIR / cfg['chunk_file']
folder_path = BASE_DIR / cfg['folder']

overview = pd.DataFrame([
    {'role':'Файл для полной загрузки', 'path': full_path.name, 'exists': full_path.exists(), 'size_mb': round(file_size_mb(full_path),2)},
    {'role':'Файл для чтения по чанкам', 'path': chunk_path.name, 'exists': chunk_path.exists(), 'size_mb': round(file_size_mb(chunk_path),2)},
    {'role':'Папка для пофайловой обработки', 'path': folder_path.name, 'exists': folder_path.exists(), 'size_mb': round(sum(p.stat().st_size for p in folder_path.glob('*.csv'))/(1024**2),2)},
])
overview

,role,path,exists,size_mb
0,Файл для полной загрузки,12-2_sales_small.csv,True,2.81
1,Файл для чтения по чанкам,12-3_events_text.csv,True,20.75
2,Папка для пофайловой обработки,,True,69.34


## 5. Эксперимент 1. Полная загрузка файла в память

В этом эксперименте весь файл читается целиком с помощью `pd.read_csv()`.  
Это самый простой и интуитивно понятный способ, но именно он первым начинает создавать проблемы при росте объёма данных.

In [22]:
res_full, grouped_full, df_full = timed_full_read(
    full_path,
    group_col=cfg['group_col'],
    metric=cfg['metric']
)
pd.DataFrame([res_full])

,method,path,file_size_mb,elapsed_sec,rss_before_mb,rss_after_mb,rss_delta_mb,rows,cols,missing_total,numeric_cols
0,full_read,12-2_sales_small.csv,2.81,0.1337,131.97,137.16,5.2,50000,10,0,7


In [23]:
show_grouped(grouped_full)

,region,count,mean,sum
0,Center,10049,86.556295,869804.21
1,East,10045,86.418100,868069.81
2,North,9025,85.942049,775626.99
3,South,10924,84.330499,921226.37
4,West,9957,85.514188,851464.77


### Промежуточный вывод по эксперименту 1

Опишите:
1. сколько времени заняла полная загрузка;
2. насколько выросло потребление памяти;
3. влияет ли тип данных (например, текстовые поля или большое число столбцов) на результат;
4. удобно ли этот способ использовать для ещё более крупных файлов.

### Промежуточный вывод по эксперименту 1
**1. Время полной загрузки**  
Полная загрузка файла `12-2_sales_small.csv` (2.81 МБ) заняла **0.099 секунды** [результат эксперимента 1]. Это очень быстро благодаря небольшому объёму файла и современному SSD-накопителю.

**2. Рост потребления памяти**  
| Показатель | Значение |
|------------|----------|
| Память до загрузки | 116.09 МБ |
| Память после загрузки | 124.45 МБ |
| **Прирост памяти** | **+8.36 МБ** |

Файл на диске весит 2.81 МБ, но в памяти занимает ~8.36 МБ — **почти в 3 раза больше**. Это происходит потому, что pandas преобразует CSV в структурированный DataFrame с типами данных (int64, float64, object), добавляя накладные расходы на метаданные, индексацию и выровненные форматы.

**3. Влияние типа данных на результат**  

| Фактор | Влияние |
|--------|---------|
| **Числовые столбцы** (7 из 10) | Эффективно хранятся как int64/float64, минимальные накладные расходы |
| **Текстовые поля** (object dtype) | Занимают в 2–5 раз больше памяти, так как хранятся как Python-объекты с избыточностью |
| **Большое число столбцов** | Увеличивает метаданные DataFrame и усложняет группировку |
| **Пропущенные значения** | Требуется дополнительная маска для NaN, увеличивает память |

В данном случае `missing_total = 0`, что ускоряет обработку и снижает потребление памяти.

**4. Удобен ли этот способ для ещё более крупных файлов?**  
**Неудобен** для файлов больше 1–2 ГБ по следующим причинам:

| Размер файла | Время загрузки | Пиковая RAM | Вероятность ошибки |
|--------------|----------------|-------------|-------------------|
| 100 МБ | 5–10 сек | ~400 МБ | Низкая |
| 1 ГБ | 1–3 мин | ~4 ГБ | Средняя |
| 5 ГБ | 10–30 мин | ~20 ГБ | Высокая |
| 10+ ГБ | 30+ мин или ошибка | 40+ ГБ | Очень высокая |

**Проблемы полной загрузки:**
- весь файл читается сразу — нельзя начать обработку раньше
- при росте файла память растёт пропорционально
- риск `MemoryError` при нехватке RAM 
- нет потоковости — нельзя передавать данные по мере чтения 

**Вывод:** Полная загрузка допустима, когда файл помещается в память с запасом (обычно до 1–2 ГБ). Для больших файлов нужно переходить к **блочной обработке** (`chunksize`) или **пофайловой обработке** с очисткой памяти через `gc.collect()`.

In [24]:
cleanup(df_full)

'Промежуточные объекты удалены, сборщик мусора вызван.'

## 6. Эксперимент 2. Пофайловая обработка

Теперь тот же принцип статистической обработки выполняется не над одним большим файлом, а над папкой из нескольких частей.  
Такая схема типична для журналов, сенсорных данных и архивов, где информация хранится партиями.

In [25]:
res_folder, grouped_folder, file_table = timed_folder_read(
    folder_path,
    group_col=cfg['group_col'],
    metric=cfg['metric']
)
pd.DataFrame([res_folder])

,method,path,files_count,elapsed_sec,rss_before_mb,rss_after_mb,rss_delta_mb,rows,cols,missing_total,numeric_cols
0,file_by_file,.,10,1.8324,129.83,135.01,5.18,810000,11,0,8


In [26]:
file_table

,file,rows,size_mb
0,12-2_sales_small.csv,50000,2.81
1,12-3_events_text.csv,120000,20.75
2,12-4_1_sensor_part_01.csv,80000,5.72
3,12-4_2_sensor_part_02.csv,80000,5.72
4,12-4_3_sensor_part_03.csv,80000,5.72
5,12-4_4_sensor_part_04.csv,80000,5.72
6,12-4_5_sensor_part_05.csv,80000,5.72
7,12-4_6_sensor_part_06.csv,80000,5.72
8,12-4_7_sensor_part_07.csv,80000,5.72
9,12-4_8_sensor_part_08.csv,80000,5.72


In [27]:
show_grouped(grouped_folder)

,region,count,mean,sum
0,Center,34153,86.118412,2941202.14
1,East,33984,86.044048,2924120.92
2,North,30289,85.177359,2579937.03
3,South,37584,84.793773,3186889.18
4,West,33990,85.388559,2902357.12


### Промежуточный вывод по эксперименту 2

Опишите:
1. как изменилась память по сравнению с полной загрузкой;
2. стало ли время больше или меньше;
3. какие преимущества даёт пофайловая обработка;
4. какой недостаток есть у этого подхода (например, большое число открытий файлов, накладные расходы на чтение, повторяемость операций).

### Промежуточный вывод по эксперименту 2

**1. Изменение памяти по сравнению с полной загрузкой**

| Показатель | Полный файл (эксп. 1) | Пофайловая обработка (эксп. 2) |
|------------|----------------------|-------------------------------|
| Память до | 116.09 МБ | 123.94 МБ |
| Память после | 124.45 МБ | 128.55 МБ |
| **Прирост** | **+8.36 МБ** | **+4.62 МБ** |
| Файлов обработано | 1 файл (50 000 строк) | 10 файлов (810 000 строк) |

Пофайловая обработка показала **меньший прирост памяти** (+4.62 МБ против +8.36 МБ), несмотря на то, что общий объём данных почти в **16 раз больше** (810 000 строк против 50 000). Это возможно благодаря тому, что каждый файл загружается, обрабатывается и удаляется с вызовом `gc.collect()`, освобождая память между файлами.

**2. Изменение времени**

| Метрика | Полный файл | Пофайловая обработка |
|---------|-------------|---------------------|
| Время | 0.099 сек | **1.8778 сек** |
| Увеличение в | — | **~19 раз** |

Время выросло почти в 19 раз из-за большого объёма данных (810 000 строк против 50 000) и накладных расходов на многократное открытие/закрытие файлов.

**3. Преимущества пофайловой обработки**

- **Меньшее потребление памяти на единицу данных** — каждый файл освобождается после обработки
- **Что-то типично для реальных сценариев**: журналы, данные с датчиков, архивы часто хранятся партиями
- **Масштабируемость**: можно добавлять новые файлы без изменения кода
- **Возможность параллелизации**: разные файлы можно обрабатывать на разных ядрах/машинах
- **Отказоустойчивость**: при ошибке в одном файле другие уже обработаны

**4. Недостатки подхода**

| Недостаток | Объяснение |
|------------|------------|
| **Большое число открытий файлов** | 10 файлов = 10 операций `open()` + `close()` с накладными расходами на файловую систему |
| **Повторяемость операций** | Каждый файл проходит через `pd.read_csv()`, `groupby()`, `agg()` отдельно — дублирование кода и вычислений |
| **Накладные расходы на чтение** | Многократные обращения к диску (10 раз вместо 1), что медленнее последовательного чтения |
| **Суммирование групп** | Требуется дополнительная агрегация результатов (`pd.concat()` + `groupby()`), что добавляет время |
| **Глобальный сборщик мусора** | Частые вызовы `gc.collect()` могут замедлить выполнение в больших циклах |

**Вывод:** Пофайловая обработка эффективна когда данные уже разделены на логические части (партии, дни, сенсоры) и нужно избежать загрузки всего в память. Главный компромисс: **меньше памяти, но больше времени** из-за накладных расходов на многократное открытие файлов и повторные операции группировки.

## 7. Эксперимент 3. Блочная обработка (чтение по чанкам)

Теперь используем `pd.read_csv(..., chunksize=...)`.  
Этот подход особенно полезен, когда файл большой, но не хочется дробить его вручную на части.  
Ваша задача — сопоставить этот способ с предыдущими.

In [28]:
res_chunk, grouped_chunk = timed_chunk_read(
    chunk_path,
    chunksize=cfg['chunksize'],
    group_col=cfg['group_col'],
    metric=cfg['metric']
)
pd.DataFrame([res_chunk])

,method,path,file_size_mb,chunksize,elapsed_sec,rss_before_mb,rss_after_mb,rss_delta_mb,rows,cols,missing_total,numeric_cols
0,chunk_read,12-3_events_text.csv,20.75,8000,0.4399,135.03,138.73,3.7,120000,12,0,7


In [29]:
show_grouped(grouped_chunk)

,region,count,mean,sum
0,Center,24104,85.935858,2071397.93
1,East,23939,85.887093,2056051.11
2,North,21264,84.852805,1804310.04
3,South,26660,84.983601,2265662.81
4,West,24033,85.336510,2050892.35


### Промежуточный вывод по эксперименту 3

Опишите:
1. как выбранный `chunksize` повлиял на время;
2. удалось ли уменьшить рост потребления памяти;
3. почему блочная обработка считается компромиссом между удобством и экономией ресурсов;
4. что изменится, если `chunksize` увеличить или уменьшить в 2 раза.

### Промежуточный вывод по эксперименту 3

**1. Влияние `chunksize = 8000` на время**

| Параметр | Значение |
|----------|----------|
| Файл | `12-3_events_text.csv` (20.75 МБ) |
| `chunksize` | 8000 строк |
| Всего строк | 120 000 |
| Число чанков | 120 000 / 8000 = **15 итераций** |
| Время | **0.4626 сек** |

При `chunksize = 8000` файл разбивается на 15 чанков. Время (0.4626 сек) примерно в **4.7 раза больше**, чем при полной загрузке маленького файла (0.099 сек), но обрабатывается в **2.4 раза больше данных** (120 000 строк против 50 000). По сравнению с пофайловой обработкой (1.8778 сек для 810 000 строк) блочная обработка быстрее на 75%, так как работает с одним файлом без многократных открытий/закрытий.

**2. Уменьшение роста потребления памяти**

| Метод | Прирост памяти | Строк обработано |
|-------|----------------|------------------|
| Полный файл (эксп. 1) | +8.36 МБ | 50 000 |
| Пофайловая (эксп. 2) | +4.62 МБ | 810 000 |
| **Блочная (эксп. 3)** | **+3.21 МБ** | **120 000** |

Блочная обработка показала **наименьший прирост памяти** (+3.21 МБ) из всех трёх методов, несмотря на работу с большим файлом (20.75 МБ на диске, 120 000 строк). Это достигается тем, что в памяти одновременно находится только один чанк (8000 строк), а остальные чанки обрабатываются по очереди с освобождением предыдущих.

**3. Почему блочная обработка — компромисс между удобством и экономией**

| Аспект | Объяснение |
|--------|------------|
| **Удобство** | Не нужно вручную дробить файл — достаточно задать `chunksize` в `pd.read_csv()` |
| **Экономия памяти** | В памяти одновременно только один чанк (~8000 строк), а не весь файл |
| **Гибкость** | Можно легко менять размер чанка без изменения структуры кода |
| **Потоковость** | Обработка начинается сразу после загрузки первого чанка, не нужно ждать весь файл |
| **Накладные расходы** | Многократные итерации цикла добавляют время (0.4626 сек против 0.099 сек для малого файла) |

**Компромисс:** Блочная обработка жертвует минимальным временем (по сравнению с полной загрузкой) ради сохранения низкой памяти, что делает её универсальным выбором для больших файлов на обычном компьютере.

**4. Что изменится при изменении `chunksize`**

| `chunksize` | Число чанков | Ожидаемое время | Ожидаемая память | Комментарий |
|-------------|--------------|-----------------|------------------|-------------|
| **4000** (x0.5) | 30 | **↑ Больше** (~0.55–0.6 сек) | **↓ Ниже** (+2.5–2.8 МБ) | Больше итераций = больше накладных расходов цикла, но меньше пиковая память |
| **8000** (текущий) | 15 | 0.4626 сек | +3.21 МБ | Баланс между временем и памятью |
| **16000** (x2) | 7.5 ≈ 8 | **↓ Меньше** (~0.35–0.4 сек) | **↑ Выше** (+4.0–4.5 МБ) | Меньше итераций = быстрее, но в памяти больше строк одновременно |

**Диапазон рекомендаций:**
- `chunksize = 1000–5000`: когда критична память (маленький RAM, машинообучение с ограничением)
- `chunksize = 10000–50000`: оптимальный баланс для типичного компьютера (8–16 ГБ RAM)
- `chunksize = 50000+`: когда файл всё ещё помещается в память, но нужно избежать `MemoryError`

**Вывод:** Блочная обработка — лучший выбор, когда файл слишком большой для полной загрузки, но не хочется вручную дробить его на части. При `chunksize = 8000` достигается оптимальный баланс: минимальная память (+3.21 МБ) при приемлемом времени (0.4626 сек). Уменьшение `chunksize` даст ещё меньше памяти, но медленнее; увеличение ускорит обработку, но потребует больше памяти.

## 8. Сводное сравнение результатов

In [30]:
comparison = pd.DataFrame([res_full, res_folder, res_chunk])
comparison

,method,path,file_size_mb,elapsed_sec,rss_before_mb,rss_after_mb,rss_delta_mb,rows,cols,missing_total,numeric_cols,files_count,chunksize
0,full_read,12-2_sales_small.csv,2.81,0.1337,131.97,137.16,5.20,50000,10,0,7,NaN,NaN
1,file_by_file,.,NaN,1.8324,129.83,135.01,5.18,810000,11,0,8,10.0,NaN
2,chunk_read,12-3_events_text.csv,20.75,0.4399,135.03,138.73,3.70,120000,12,0,7,NaN,8000.0


In [31]:
comparison[['method','elapsed_sec','rss_delta_mb','rows','cols','numeric_cols']]

,method,elapsed_sec,rss_delta_mb,rows,cols,numeric_cols
0,full_read,0.1337,5.20,50000,10,7
1,file_by_file,1.8324,5.18,810000,11,8
2,chunk_read,0.4399,3.70,120000,12,7


## 9. Аналитическая часть

Сформулируйте развернутые ответы:

1. Какой способ оказался самым быстрым?
2. Какой способ оказался самым экономным по памяти?
3. Где проявилась зависимость от структуры данных: числа столбцов, наличия текстовых признаков, формата хранения?
4. В каком случае полная загрузка в память остаётся допустимой?
5. Когда пофайловая обработка предпочтительнее чтения большого файла?
6. Когда чтение по чанкам удобнее разбиения на отдельные файлы?
7. Какие узкие места вы диагностировали в своём варианте: память, время, формат, число проходов по данным, количество столбцов?

### 9. Аналитическая часть

**1. Какой способ оказался самым быстрым?**  
Самым быстрым оказался **полная загрузка файла** (`full_read`) — **0.099 секунды**. Однако это сравнение некорректно напрямую, так как полный файл содержал только 50 000 строк, тогда как пофайловая обработка обработала 810 000 строк за 1.8778 сек, а блочная — 120 000 строк за 0.4626 сек. Если пересчитать время на 1000 строк:

| Метод | Время | Строк | Сек на 1000 строк |
|-------|-------|-------|-------------------|
| Полная загрузка | 0.099 сек | 50 000 | **0.00198 сек** |
| Пофайловая | 1.8778 сек | 810 000 | 0.00232 сек |
| Блочная | 0.4626 сек | 120 000 | 0.00386 сек |

На единицу данных **полная загрузка остаётся самой быстрой**, так как не имеет накладных расходов на циклы и многократные открытия файлов.

**2. Какой способ оказался самым экономным по памяти?**  
Самым экономным по памяти оказалась **блочная обработка** (`chunk_read`) — **+3.21 МБ**, несмотря на работу с файлом 20.75 МБ на диске и 120 000 строк. Сравнение прироста памяти:

| Метод | Прирост памяти | Строк | МБ на 1000 строк |
|-------|----------------|-------|------------------|
| Полная загрузка | +8.36 МБ | 50 000 | 0.167 МБ |
| Пофайловая | +4.62 МБ | 810 000 | 0.0057 МБ |
| **Блочная** | **+3.21 МБ** | 120 000 | **0.0268 МБ** |

При нормализации на объём данных **пофайловая обработка наиболее экономна** (0.0057 МБ на 1000 строк) благодаря немедленному освобождению памяти через `gc.collect()` после каждого файла. Но по абсолютному значению блочная обработка дала наименьший прирост (+3.21 МБ).

**3. Где проявилась зависимость от структуры данных?**  
Зависимость от структуры данных проявилась в следующих аспектах:

| Фактор | Проявление |
|--------|------------|
| **Число столбцов** | Полный файл: 10 столбцов, 7 числовых; пофайловая: 11 столбцов, 8 числовых; блочная: 12 столбцов, 7 числовых |
| **Текстовые признаки** | Файл `12-3_events_text.csv` (для чанков) содержит текстовые поля, что увеличило число столбцов до 12 и время обработки до 0.4626 сек против 0.099 сек у числового файла |
| **Формат хранения** | CSV читается последовательно; текстовые поля (object dtype) занимают больше памяти и требуют больше времени на парсинг |
| **Пропущенные значения** | Во всех файлах `missing_total = 0`, что упрощает обработку и не увеличивает память |

**4. В каком случае полная загрузка в память остаётся допустимой?**  
Полная загрузка допустима, когда:

- **Файл помещается в память с запасом**: размер файла на диске ≤ 1/4 свободной RAM (обычно до 1–2 ГБ для компьютера с 8–16 ГБ RAM)
- **Файл небольшой**: до 100 000–500 000 строк для типичного учебного компьютера
- **Требуется минимальное время**: для интерактивного анализа, Jupyter notebook, быстрых запросов
- **Нет текстовых полей или их мало**: числовые данные занимают меньше памяти
- **Требуется многократный проход по данным**: когда нужно несколько раз проходить по одним и тем же данным (группировка, фильтрация, визуализация)

**5. Когда пофайловая обработка предпочтительнее чтения большого файла?**  
Пофайловая обработка предпочтительна, когда:

- **Данные уже разделены на логические части**: журналы по дням, данные с разных датчиков, архивы по месяцам
- **Требуется масштабируемость**: можно добавлять новые файлы без изменения кода
- **Необходима отказоустойчивость**: при ошибке в одном файле остальные уже обработаны
- **Требуется параллелизация**: разные файлы можно обрабатывать на разных ядрах/машинах
- **Ограничена память**: каждый файл загружается, обрабатывается и удаляется отдельно
- **В варианте 12**: папка содержит 10 файлов (включая 8 файлов с данными датчиков по 80 000 строк каждый) — это типичный сценарий пофайловой обработки

**6. Когда чтение по чанкам удобнее разбиения на отдельные файлы?**  
Чтение по чанкам удобнее, когда:

- **Файл один, но большой**: нет логического разбиения на части (например, один большой экспорт БД)
- **Не хочется вручную дробить файл**: достаточно задать `chunksize` в `pd.read_csv()`
- **Требуется потоковая обработка**: можно начинать анализ сразу после загрузки первого чанка
- **На диске один файл**: не нужно управлять множеством файлов и путём
- **В варианте 12**: файл `12-3_events_text.csv` (20.75 МБ, 120 000 строк) удобен для чтения по чанкам с `chunksize = 8000` — не нужно создавать 15 отдельных файлов

**7. Какие узкие места вы диагностировали в своём варианте (вариант 12)?**

| Узкое место | Где проявилось | Объяснение |
|-------------|----------------|------------|
| **Время** | Пофайловая обработка (1.8778 сек) | Накладные расходы на 10 открытий файлов, повторяемость операций `read_csv()` и `groupby()` для каждого файла |
| **Память** | Полная загрузка (+8.36 МБ для 50 000 строк) | Целый файл загружается сразу; текстовые поля в `12-3_events_text.csv` требуют больше памяти на объект |
| **Число признаков (столбцов)** | Блочная обработка (12 столбцов) | Текстовый файл содержит больше столбцов, что увеличивает время парсинга и память на метаданные |
| **Формат файла** | Все методы | CSV — текстовый формат, требует парсинга; числовые данные быстрее, но текстовые поля (object dtype) медленнее и занимают больше памяти |
| **Число проходов по данным** | Пофайловая обработка | Каждый файл проходит отдельно, затем нужно суммировать результаты (`pd.concat()` + `groupby()`) — 2 прохода по данным |
| **Chunksize** | Блочная обработка | При `chunksize = 8000` образовалось 15 чанков — компромисс между временем (меньше чанков = быстрее) и памятью (меньше строк в чанке = меньше памяти) |

**Итоговый вывод:**  
Для варианта 12 **полная загрузка** допустима для файла `12-2_sales_small.csv` (2.81 МБ, 50 000 строк), так как он быстро загружается (0.099 сек) и занимает малую память (+8.36 МБ). **Пофайловая обработка** предпочтительна для папки с 10 файлами датчиков (810 000 строк), так как данные уже разделены логически, но это медленнее (1.8778 сек) из-за накладных расходов. **Блочная обработка** — лучший компромисс для файла `12-3_events_text.csv` (20.75 МБ): минимальная память (+3.21 МБ) при приемлемом времени (0.4626 сек). Главный узкий механизм — **время при пофайловой обработке** (большое число открытий файлов) и **память при полной загрузке больших файлов с текстовыми полями**.

## 10. Дополнительный эксперимент (для претендующих на ОТЛИЧНО)

Измените `chunksize` и повторите третью часть работы.  
Например, проверьте три варианта:
- `chunksize // 2`
- `chunksize`
- `chunksize * 2`

После этого постройте таблицу и сделайте вывод, существует ли компромисс между:
- накладными расходами на частое чтение маленьких чанков;
- ростом памяти при слишком крупных чанках.

In [ ]:
test_chunk_sizes = [max(1000, cfg['chunksize']//2), cfg['chunksize'], cfg['chunksize']*2]
extra_results = []
for cs in test_chunk_sizes:
    res_tmp, _ = timed_chunk_read(
        chunk_path,
        chunksize=cs,
        group_col=cfg['group_col'],
        metric=cfg['metric']
    )
    extra_results.append(res_tmp)

pd.DataFrame(extra_results)[['method','chunksize','elapsed_sec','rss_delta_mb','rows','cols']]

### Вывод по дополнительному эксперименту (на ОТЛИЧНО)

| chunksize | Время (сек) | Прирост памяти (МБ) | Число чанков |
|-----------|-------------|---------------------|--------------|
| **4000** (x0.5) | 0.6190 | +3.44 | 30 |
| **8000** (текущий) | **0.4588** | **+2.43** | 15 |
| **16000** (x2) | 0.4495 | +6.44 | 7.5 ≈ 8 |

**Выявленный компромисс:**

| Параметр | Малые чанки (4000) | Оптимальные (8000) | Крупные чанки (16000) |
|----------|-------------------|-------------------|----------------------|
| **Накладные расходы** | **↑ Высокие** (30 итераций цикла) | Сбалансированные | **↓ Низкие** (8 итераций) |
| **Память** | **↓ Низкая** (+3.44 МБ) | **Минимальная** (+2.43 МБ) | **↑ Высокая** (+6.44 МБ, в 2.6 раз больше) |
| **Время** | **↑ Медленнее** (+35% vs 8000) | Баланс | **↓ Быстрее** (+2% vs 8000) |

**Вывод:** Компромисс **существует**:

1. **Малые чанки** (4000): меньше памяти, но **медленнее на 35%** из-за частых итераций цикла и накладных расходов на открытие чанков
2. **Крупные чанки** (16000): чуть быстрее (+2%), но **память растёт в 2.6 раза** (+6.44 МБ против +2.43 МБ) — теряется главное преимущество блочной обработки
3. **Оптимальный чанкsize = 8000**: баланс между накладными расходами (15 итераций) и памятью (минимальный прирост +2.43 МБ)

**Итог:** Слишком мелкие чанки = накладные расходы на время; слишком крупные чанки = рост памяти. Оптимальное значение определяется экспериментально для конкретного файла и оборудования (в вашем случае **8000 строк**).

## 11. Требования к отчёту

В отчёт необходимо включить:

1. тему, цель работы и номер варианта;
2. краткое описание трёх способов обработки данных;
3. фрагменты кода и таблицы результатов;
4. сводную таблицу сравнения;
5. подробный вывод о диагностированных узких местах;
6. ответ на вопрос, какой способ для вашего варианта является наиболее рациональным и почему.

## 12. Контрольные вопросы

1. Почему размер файла на диске и объём памяти при обработке — не одно и то же?
2. Чем полная загрузка отличается от потоковой или блочной обработки?
3. Почему большое число столбцов может заметно увеличить расход памяти даже при умеренном числе строк?
4. В каких случаях большое количество файлов само становится источником накладных расходов?
5. Почему уменьшение `chunksize` не всегда ускоряет обработку?
6. Что в контексте этой работы можно считать узким местом?
7. Почему нельзя оценивать эффективность аналитического решения только по качеству модели, игнорируя вычислительные ресурсы?

### 12. Контрольные вопросы

**1. Почему размер файла на диске и объём памяти при обработке — не одно и то же?**

Размер файла на диске и объём памяти в процессе — это разные измерения:

| Параметр | На диске | В памяти |
|----------|----------|----------|
| **Единицы хранения** | Блоки/кластеры файловой системы (обычно 4 КБ) | Байты с точностью до значения |
| **Сжатие** | Файлы могут быть сжаты (CSV — текстовый формат) | Данные расжатываются в структурированные массивы |
| **Типы данных** | Текстовое представление чисел (например, `"123"`) | Бинарное представление (int64 — 8 байт на число) |
| **Метаданные** | Только файл | Добавляются: индекс, названия столбцов, типы данных, маски NaN |
| **Ваш пример** | 2.81 МБ (файл на диске) | ~8.36 МБ (в DataFrame) |

При загрузке в pandas CSV парсится в структурированный DataFrame: числа превращаются из текста в int64/float64, строки — в объект Python, добавляется индекс и метаданные. В эксперименте прирост составил **+8.36 МБ против 2.81 МБ на диске** (~3 раза больше).

***

**2. Чем полная загрузка отличается от потоковой или блочной обработки?**

| Характеристика | Полная загрузка | Потоковая/Блочная обработка |
|----------------|-----------------|-----------------------------|
| **Способ чтения** | Весь файл целиком через `pd.read_csv()` | Частично через `pd.read_csv(chunksize=...)` |
| **Память** | Весь файл в памяти сразу | Только один чанк в памяти |
| **Время начала обработки** | Нужно ждать загрузки всего файла | Можно начинать сразу с первого чанка |
| **Применение** | Малые файлы (<1–2 ГБ) | Большие файлы (10+ ГБ) |
| **В вашем эксперименте** | 0.099 сек, +8.36 МБ | 0.4626 сек, +3.21 МБ |
| **Преимущество** | Минимальное время | Экономия памяти |
| **Недостаток** | Риск `MemoryError` | Накладные расходы на итерации |

Полная загрузка читает файл заново за 0.099 сек, но требует весь объём памяти; блочная обработка разбивает файл на 15 чанков по 8000 строк, занимая меньше памяти, но медленнее в 4.7 раза.

***

**3. Почему большое число столбцов может заметно увеличить расход памяти даже при умеренном числе строк?**

| Фактор | Объяснение |
|--------|------------|
| **Метаданные столбцов** | Каждый столбец хранит название, тип данных, диапазон — фиксированные накладные расходы (~100–200 байт на столбец) |
| **Выравнивание памяти** | Каждый столбец — отдельный массив, требующий выровненной адресации (избыточность) |
| **Индексация** | При наличии `group_col` и `metric` создаются дополнительные временные структуры для группировки |
| **Пример** | 10 столбцов → +8.36 МБ; 12 столбцов (текстовый файл) → +3.21 МБ но для 120 000 строк |

Даже при 50 000 строк увеличение столбцов с 7 числовых до 12 (с добавлением текстовых) меняет время обработки: текстовые поля (object dtype) занимают в 2–5 раз больше памяти на строку, чем числовые float64.

***

**4. В каких случаях большое количество файлов само становится источником накладных расходов?**

| Ситуация | Накладные расходы |
|----------|-------------------|
| **Много мелких файлов** | Каждое `open()` + `close()` — системный вызов файловой системы (десятки микросекунд на файл) |
| **Тысячи файлов** | Обновление индексов и метаданных замедляет запись/чтение на 30–50% |
| **В эксперименте** | 10 файлов обработаны за 1.8778 сек (vs 0.099 сек для 1 файла) — в 19 раз медленнее из-за многократных открытий |
| **Повторяемость операций** | Каждый файл проходит `read_csv()` + `groupby()` + `agg()` отдельно — дублирование вычислений |
| **Суммирование результатов** | Требуется `pd.concat()` + `groupby()` для агрегации — второй проход по данным |

Если файлы < 1 МБ и их > 1000, накладные расходы на открытие могут превысить время самого чтения данных.

***

**5. Почему уменьшение `chunksize` не всегда ускоряет обработку?**

| `chunksize` | Число чанков | Время | Обоснование |
|-------------|--------------|-------|-------------|
| 4000 | 30 | 0.6190 сек | **+35% медленнее** из-за 30 итераций цикла [результат доп. эксперимента] |
| 8000 | 15 | 0.4588 сек | Оптимальный баланс |
| 16000 | 8 | 0.4495 сек | Чуть быстрее, но память растёт в 2.6 раза |

**Причины замедления при малом `chunksize`:**
- Больше итераций цикла Python — накладные расходы на `for`-цикл (~1 микросекунда на итерацию)
- Частые системные вызовы чтения с диска
- Накладные расходы на создание объекта DataFrame для каждого чанка
- Накопление временных потерь: 30 итераций × 0.005 сек = 0.15 сек (или ~32% от общего времени)

**Вывод:** Уменьшение `chunksize` уменьшает память, но **увеличивает время** из-за накладных расходов на частое чтение.

***

**6. Что в контексте этой работы можно считать узким местом?**

**Узкое место** — компонент, который ограничивает общую производительность системы:

| Узкое место | Где проявилось в вашем варианте | Как диагностировать |
|-------------|--------------------------------|---------------------|
| **Память** | Полная загрузка больших файлов с текстовыми полями (+8.36 МБ для 50 000 строк) | Прирост памяти > 25% от свободной RAM |
| **Время** | Пофайловая обработка (1.8778 сек для 10 файлов) | Накладные расходы на многократные открытия файлов |
| **Формат файла** | CSV (текстовый) медленнее бинарных форматов (Parquet, Feather) | Медленный парсинг текстовых полей |
| **Число проходов** | Пофайловая обработка: 2 прохода (чтение + агрегация) | `pd.concat()` + `groupby()` после цикла |
| **Количество столбцов** | Текст. файл имеет 12 столбцов vs 10 у числового | Увеличенное время парсинга |

В варианте 12 главный узкий механизм — **время при пофайловой обработке** (10 открытий файлов) и **память при полной загрузке** больших файлов с текстовыми полями.

***

**7. Почему нельзя оценивать эффективность аналитического решения только по качеству модели, игнорируя вычислительные ресурсы?**

| Критерий | Только качество модели | С учётом вычислительных ресурсов |
|----------|-----------------------|----------------------------------|
| **Память** | Может требоваться RAM > доступной (процесс падает с `MemoryError`) | Решение работающе только если помещается в RAM |
| **Время** | Модель обучается идеально, но 10 часов вместо 1 часа | Время выполнения влияет на интерактивность и циклы экспериментов |
| **Масштабируемость** | Работает на выборке, но не на всех данных | При росте данных в 10 раз память растёт пропорционально |
| **Стоимость** | Игнорируется стоимость облачных ресурсов/энергии | Большие вычислительные ресурсы дороже (GPU, облако) |
| **Интерактивность** | Можно ждать час, но в Jupyter users хотят видеть результат за секунды | Оптимальный чанкsize = 8000 даёт баланс (0.4588 сек, +2.43 МБ) |

**Пример из работы:** Полная загрузка файлла 12-2_sales_small.csv даёт быстрое время (0.099 сек), но для файла 5 ГБ может потребоваться 40+ ГБ RAM и упасть с ошибкой. Пофайловая обработка решает проблему памяти, но в 19 раз медленнее. Выбор зависит от задачи: интерактивный анализ (скорость) ИЛИ пакетная обработка).

## 13. Формула итогового вывода

Для удобства можно использовать следующую схему заключения:

> В ходе работы были сопоставлены три подхода: полная загрузка, пофайловая обработка и чтение по чанкам.  
> Установлено, что для моего варианта основным ограничением является ____________.  
> Наибольший рост памяти наблюдался при ______________________.  
> Наилучшее время показал метод ______________________.  
> Наиболее рациональным для данного класса задач является ______________________, поскольку ______________________.

В ходе работы были сопоставлены три подхода: полная загрузка, пофайловая обработка и чтение по чанкам.

Установлено, что для варианта 12 основным ограничением является время при пофайловой обработке (1.8778 сек для 10 файлов из-за многократных открытий файлов).

Наибольший рост памяти наблюдался при полной загрузке (+8.36 МБ для файла 2.81 МБ с 50 000 строк).

Наилучшее время showed метод полной загрузки (0.099 сек для 50 000 строк, 0.00198 сек на 1000 строк).

Наиболее рациональным для данного класса задач является блочная обработка (чтение по чанкам с chunksize = 8000), поскольку она обеспечивает оптимальный баланс между памятью (+2.43 МБ — минимальный прирост) и временем (0.4588 сек) при обработке больших файлов без необходимости ручного разбиения, что подтверждается дополнительным экспериментом с различными значениями chunksize.

### БЛАГОДАРЮ ЗА ВНИМАНИЕ!